In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import pyfof
np.float = float # to avoid deprecation warning with pyfof

import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.animation as animation
from matplotlib import rc

matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
# plt.style.use('dark_background')
rc('animation', html = 'jshtml')
matplotlib.rcParams['animation.embed_limit'] = 2**128
plt.style.use('./plots/desi.mplstyle')

import concurrent.futures
import time

import umap
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.manifold import trustworthiness

In [ ]:
output_dir = './data/sample'

cmap = sns.color_palette('mako', as_cmap=True).reversed()
col = cmap(np.linspace(0.3, 0.8, 3))
cmap = ListedColormap(col)
cmap

In [ ]:
tiles, nights = ['2436','5568','10256'], ['20211031','20211130','20211110']

### Read data

In [ ]:
def read_spectra_file(file, wave_dir, night, tile_id, band):
    try:
        df_temp = pd.read_hdf(file, 'df')
    except Exception as e:
        print(f'Error reading {file}: {e}')
        return None
    parts = file.stem.split('_')
    if len(parts) >= 3:
        petal_id = parts[2]
        wave_file = wave_dir / f'{night}_{tile_id}_{petal_id}_wave_{band}.h5'
        if wave_file.exists():
            try:
                wave_vector = pd.read_hdf(wave_file, 'df')['WAVE'].values
                df_temp['WAVE'] = [wave_vector] * len(df_temp)
            except Exception as e:
                print(f'Error reading wave file {wave_file}: {e}')
        else:
            print(f'Wave file not found: {wave_file}')
    return df_temp

In [ ]:
def pad_flux_arrays(flux_series):
    arrs = flux_series.tolist()
    max_length = max((len(x) for x in arrs if isinstance(x, (list, np.ndarray))), default=0)
    padded = [np.pad(x, (0, max_length - len(x)), constant_values=0)
              if isinstance(x, (list, np.ndarray)) else x for x in arrs]
    return pd.Series(padded, index=flux_series.index)

In [ ]:
def get_data(tile_id, night, output_dir, band='brz', umap_params=None,
                      linking_length=0.35, min_cluster_size=7):
    output_dir = Path(output_dir)
    spectra_dir = output_dir / tile_id / night / 'spectra'
    wave_dir = output_dir / tile_id / night / 'wave'
    spectra_files = list(spectra_dir.glob(f'{night}_{tile_id}_*_{band}_spectra.h5'))

    if not spectra_files:
        print(f'No files in {spectra_dir}')
        return None

    with concurrent.futures.ThreadPoolExecutor() as executor:
        dfs = [df for df in executor.map(lambda f: read_spectra_file(f, wave_dir, night, tile_id, band),
                                           spectra_files) if df is not None]
    if not dfs:
        print('No valid data files found.')
        return None

    df = pd.concat(dfs, ignore_index=True)

    try:
        df['FLUX'] = df['FLUX'].apply(lambda x: np.fromstring(x, sep=',') if isinstance(x, str) else x)
    except Exception as e:
        print(f'Error processing FLUX: {e}')
        return None

    try:
        df['FLUX_PADDED'] = pad_flux_arrays(df['FLUX'])
        flux_matrix = np.stack(df['FLUX_PADDED'].values)
        print(f'Tile {tile_id} - Night {night}: {flux_matrix.shape[0]} spectra, FLUX shape: {flux_matrix.shape}')
    except Exception as e:
        print(f'Error padding FLUX: {e}')
        return None

    wave_matrix = None
    if 'WAVE' in df.columns:
        try:
            df['WAVE_PADDED'] = pad_flux_arrays(df['WAVE'])
            wave_matrix = np.stack(df['WAVE_PADDED'].values)
            print(f'WAVE matrix shape: {wave_matrix.shape}')
        except Exception as e:
            print(f'Error processing WAVE: {e}')
            wave_matrix = None

    X = np.hstack((flux_matrix, wave_matrix)) if wave_matrix is not None else flux_matrix
    return X, df, flux_matrix

-----

In [ ]:
for i in range(len(tiles[:1])):
    print(f'\n--- Processing tile {tiles[i]}, night {nights[i]}')
    X, df, flux_matrix = get_data(tiles[i], nights[i], output_dir, band='b', umap_params=None)

In [ ]:
df['FLUX_PADDED'].iloc[0]

-----

### Model

In [ ]:
def get_umap(X, df, flux_matrix, umap_params=None, linking_length=0.35, min_cluster_size=7):
    if umap_params is None:
        umap_params = {'n_neighbors': 45, 'min_dist': 1.0, 'metric': 'cosine', 'n_jobs': -1}
    umap_params.setdefault('n_jobs', -1)

    embedding = umap.UMAP(**umap_params).fit_transform(X)
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'])
    df_full = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)

    cluster_labels = np.full(len(embedding), -1)
    for i, group in enumerate(pyfof.friends_of_friends(embedding, linking_length)):
        if len(group) >= min_cluster_size:
            cluster_labels[np.array(group)] = i

    df_full['pyfof_cluster'] = cluster_labels
    df_full['is_outlier'] = (cluster_labels == -1)
    n_clusters = len(np.unique(cluster_labels[cluster_labels != -1]))
    outlier_count = df_full['is_outlier'].sum()
    print(f'Clusters: {n_clusters}')
    print(f'Outliers: {outlier_count}, {(outlier_count/flux_matrix.shape[0]*100):.2f}% of data')

    if 'SPECTYPE' in df_full.columns:
        df_full['SPECTYPE'] = df_full['SPECTYPE'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

    return df_full, n_clusters

In [ ]:
def plot_umap(df_full, tile_id, night, n_clusters):
    plt.figure()
    if 'SPECTYPE' in df_full.columns:
        categories = sorted(df_full['SPECTYPE'].unique())
        palette = ["#75bbfd", "#c20078", "#96f97b"]
        palette = dict(zip(categories, sns.color_palette('mako', as_cmap=True)(np.linspace(0.1, 0.8, len(categories)))))
        for i, cat in enumerate(categories):
            subset = df_full[(df_full['SPECTYPE'] == cat) & (~df_full['is_outlier'])]
            plt.scatter(subset['UMAP1'], subset['UMAP2'], subset['UMAP3'], s=2, color=palette[cat], label=cat)
    else:
        plt.scatter(df_full['UMAP1'], df_full['UMAP2'], s=2)

    outliers = df_full[df_full['is_outlier']]
    if not outliers.empty:
        plt.scatter(outliers['UMAP1'], outliers['UMAP2'], s=15, marker='x',
                    linewidths=0.8, color='red', label='Outliers')

    plt.legend(fontsize=9)
    plt.xticks([])
    plt.yticks([])
    plt.axis('off')
    plt.title(f'{X.shape[0]} Spectra, {n_clusters} clusters, {df_full["is_outlier"].sum()} outliers', y=1.05, fontsize=14)
    plt.savefig(f'./plots/umap/umap_2d_{tile_id}_{night}.png', dpi=360)
    # plt.show()
    plt.close()

In [ ]:
for i in range(len(tiles)):
    print(f'\n--- Processing tile {tiles[i]}, night {nights[i]}')
    X, df, flux_matrix = get_data(tiles[i], nights[i], output_dir, band='b', umap_params=None)
    df_full, n_clusters = get_umap(X, df, flux_matrix)
    plot_umap(df_full, tiles[i], nights[i], n_clusters)

3D

In [ ]:
def get_umap_3d(X, df, flux_matrix, umap_params=None, linking_length=0.45, min_cluster_size=7):
    if umap_params is None:
        umap_params = {'n_neighbors': 45, 'min_dist': 1.0, 'metric': 'cosine',
                       'n_jobs': -1, 'n_components':3}
    umap_params.setdefault('n_jobs', -1)

    embedding = umap.UMAP(**umap_params).fit_transform(X)
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2', 'UMAP3'])
    df_full = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)

    cluster_labels = np.full(len(embedding), -1)
    for i, group in enumerate(pyfof.friends_of_friends(embedding, linking_length)):
        if len(group) >= min_cluster_size:
            cluster_labels[np.array(group)] = i

    df_full['pyfof_cluster'] = cluster_labels
    df_full['is_outlier'] = (cluster_labels == -1)
    n_clusters = len(np.unique(cluster_labels[cluster_labels != -1]))
    outlier_count = df_full['is_outlier'].sum()
    print(f'Clusters: {n_clusters}')
    print(f'Outliers: {outlier_count}, {(outlier_count/flux_matrix.shape[0]*100):.2f}% of data')

    if 'SPECTYPE' in df_full.columns:
        df_full['SPECTYPE'] = df_full['SPECTYPE'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

    return df_full, n_clusters

In [ ]:
X, df, flux_matrix = get_data(tiles[0], nights[0], output_dir, band='b', umap_params=None)

In [ ]:
df_full, n_clusters = get_umap_3d(X, df, flux_matrix)

In [ ]:
df_full, tile_id, night = df_full, tiles[0], nights[0]
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

if 'SPECTYPE' in df_full.columns:
    categories = sorted(df_full['SPECTYPE'].unique())
    palette = dict(zip(categories, sns.color_palette('mako', as_cmap=True).reversed()(np.linspace(0.2, 0.8, len(categories)))))

    out = []
    for i, cat in enumerate(categories):
        subset = df_full[(df_full['SPECTYPE'] == cat) & (~df_full['is_outlier'])]
        out.append(ax.scatter(subset['UMAP1'], subset['UMAP2'], subset['UMAP3'], s=1,
                                color=palette[cat], label=cat, alpha=1.0))
else:
    ax.scatter(df_full['UMAP1'], df_full['UMAP2'], df_full['UMAP3'], s=4)

outliers = df_full[df_full['is_outlier']]
if not outliers.empty:
    out.append(ax.scatter(outliers['UMAP1'], outliers['UMAP2'], outliers['UMAP3'], s=3, marker='x',
                linewidths=0.5, color='red', label='Outliers'))

plt.legend(fontsize=9, loc='lower left', markerscale=2)
plt.xticks([])
plt.yticks([])
plt.axis('off')
plt.title(f'{X.shape[0]} Spectra, {n_clusters} clusters, {df_full["is_outlier"].sum()} outliers',
          y=1.05, fontsize=14)

def update(frame):
    ax.view_init(azim=frame, elev=frame/2)

animation.FuncAnimation(fig, update, frames=360, interval=120)

### Get spectra

In [ ]:
from astropy.convolution import convolve, Gaussian1DKernel

In [ ]:
bands = ['b', 'r', 'z']
color_map = {'b': 'blue', 'r': 'red', 'z': 'green'}
kernel = Gaussian1DKernel(5)

for tile, night in zip(tiles, nights):
    print(f'--- Processing tile {tile}, night {night}')
    start_time = time.time()
    fluxes = {}

    X, df, flux_matrix = get_data(tile, night, output_dir, band='brz')
    dff, n_clusters = get_umap(X, df, flux_matrix)
    plot_umap(dff, tile, night, n_clusters)
    outliers = dff[dff['is_outlier']]

    for band in bands:
        print(f'- Processing band {band}')
        X, df, flux_matrix = get_data(tile, night, output_dir, band=band)
        fluxes[band] = df

    num_outliers = len(outliers)
    for j in range(num_outliers-1):
        target_row = fluxes['b'].iloc[j]
        target_id = target_row['TARGETID']

        plt.figure(figsize=(20, 6))
        for band in bands:
            try:
                band_data = fluxes[band].iloc[j]
                plt.plot(band_data['WAVE_PADDED'], band_data['FLUX_PADDED'], color=color_map[band], alpha=0.5)
                plt.plot(band_data['WAVE_PADDED'], convolve(band_data['FLUX_PADDED'], kernel), color='k')
            except Exception as e:
                print(f'Error processing band {band} for target {target_id} in petal {target_row.get("PETAL_LOC", "Unknown")}: {e}')
                continue

        plt.xlim([3500, 9900])
        plt.xlabel(r'$\lambda$ [$\AA$]')
        spectype = target_row.get('SPECTYPE', 'Unknown').capitalize()
        petal_loc = target_row.get('PETAL_LOC', 'Unknown')
        plt.title(f'{spectype} - ID: {target_id}\nNight {night}, Tile {tile}', fontsize=20, y=1.03)
        plt.ylabel(r'$F_{\lambda}$ [$10^{-17}\ erg\ s^{-1}\ cm^{-2}\ \AA^{-1}$]')
        plt.grid(linewidth=0.5)
        plt.savefig(f'./plots/spectra/{night}/{target_id}.png', dpi=360)
        plt.close()
    print(f'----- Time taken: {time.time() - start_time:.2f} seconds\n')

### Optimize

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.manifold import trustworthiness

In [ ]:
pipeline = Pipeline([('umap', umap.UMAP(random_state=42))])

param_grid = {'umap__n_neighbors': [20, 45, 70],
              'umap__min_dist': [0.3, 0.6, 1.0],
              'umap__metric': ['euclidean','braycurtis','cosine'],
             }

def custom_score(estimator, X):
    X_embedded = estimator.named_steps['umap'].transform(X)
    trust = trustworthiness(X, X_embedded, n_neighbors=estimator.named_steps['umap'].n_neighbors)
    return trust

In [ ]:
def custom_score(estimator, X):
    X_embedded = estimator.named_steps['umap'].transform(X)
    trust = trustworthiness(X, X_embedded, n_neighbors=estimator.named_steps['umap'].n_neighbors)
    return trust

In [ ]:
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=custom_score)

In [ ]:
grid_search.fit(X)

In [ ]:
grid_search.best_params_, grid_search.best_score_